In [1]:
from transformers import T5Tokenizer
from diffusers import (
    CogVideoXPipeline,
    # CogVideoXTransformer3DModel,
    # AutoencoderKLCogVideoX,
)
from diffusers.video_processor import VideoProcessor
from diffusers.utils import export_to_video
import torch
from IPython.display import Video
# from diffusers.utils import export_to_gif
# from IPython.display import Image


In [2]:
# import torch
# from transformers import T5Tokenizer
# from diffusers import CogVideoXPipeline

def load_video_model_low_mem(model_path: str):
    # Load tokenizer
    try:
        tokenizer = T5Tokenizer.from_pretrained(
            model_path,
            subfolder="tokenizer",
            local_files_only=True
        )
        print("✅ Tokenizer loaded")
    except Exception:
        tokenizer = T5Tokenizer.from_pretrained(
            model_path,
            local_files_only=True
        )
        print("✅ Tokenizer loaded (fallback)")

    # Load pipeline with cuda
    pipe = CogVideoXPipeline.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        tokenizer=tokenizer,
        local_files_only=True,
        use_safetensors=True,
        device_map="cuda"  
    )
    
    print("✅ Pipeline loaded ")

    return pipe


# Usage
model_path = "../models/CogVideoX-5B"
pipe = load_video_model_low_mem(model_path)


✅ Tokenizer loaded


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Pipeline loaded 


In [12]:
# from diffusers.utils import export_to_video
# from diffusers.pipelines.cogvideo.pipeline_cogvideox import VideoProcessor


def generate_video(
    prompt,
    save_path,
    pipe,
    seconds=6,
    fps=6,
    steps=50,
    guidance=8.5,
    height=384,
    width=576,
    negative_prompt="blurry, deformed, low resolution, artifacts, distorted face"
):
    

    # Compute number of frames
    num_frames = seconds * fps
    if num_frames > 32:
        num_frames = 32  # safe fast limit

    print(f"Generating {num_frames} frames")

    # Run diffusion on GPU
    with torch.no_grad():
        out = pipe(
            prompt=prompt,
            num_frames=num_frames,
            height=height,
            width=width,
            num_inference_steps=steps,
            guidance_scale=guidance,
            output_type="latent",
            negative_prompt=negative_prompt,
            max_sequence_length=512
        )

    print("Decoding...")

    # Keep latents on GPU for faster decode
    latents = out.frames  # [B, F, C, H, W]

    # VAE info
    vae = pipe.vae
    vae_scale = vae.config.scaling_factor
    block_count = len(vae.config.block_out_channels)
    vae_scale_factor_spatial = 2 ** (block_count - 1)

    # Postprocessor
    video_processor = VideoProcessor(vae_scale_factor=vae_scale_factor_spatial)

    # Convert shape and unscale on GPU
    latents = latents.permute(0, 2, 1, 3, 4)   # B,C,F,H,W
    latents = latents / vae_scale

    # Decode on GPU if VAE is on GPU
    with torch.no_grad():
        decoded = vae.decode(latents).sample   # [-1,1]

    # Move to CPU only at the end
    decoded = decoded.to("cpu")

    # Final frames
    video = video_processor.postprocess_video(
        video=decoded, output_type="np"
    )[0]

    export_to_video(video, save_path, fps=fps)

    return save_path


In [85]:

prompt = """Spiderman in his red and blue suit fighting a powerful monkey on a city rooftop, dynamic action poses, fast movements, dramatic lighting, detailed textures, smooth cinematic motion, realistic environment, dust and wind effects, high energy battle scene, 4k quality, sharp focus, subtle camera shake, intense atmosphere
"""

generate_video(prompt,"videos/spider.mp4",pipe=pipe,guidance=10)



Generating 18 frames


  0%|          | 0/50 [00:00<?, ?it/s]

Decoding...


'videos/spider.mp4'

In [4]:
##Load Dataset
import pandas as pd
import os
DATASET_PATH='dataset/'
csv_path=DATASET_PATH+"test_dataset.csv"

df=pd.read_csv(csv_path)


    
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 150 non-null    int64 
 1   Title              150 non-null    object
 2   Poet               150 non-null    object
 3   text               150 non-null    object
 4   ctext              150 non-null    object
 5   Poem Link          150 non-null    object
 6   our_summary        150 non-null    object
 7   prompt_by_summary  150 non-null    object
 8   video_path         12 non-null     object
 9   prompt_by_ctext    150 non-null    object
dtypes: int64(1), object(9)
memory usage: 11.8+ KB


,id,Title,Poet,text,ctext,Poem Link,our_summary,prompt_by_summary,video_path,prompt_by_ctext
0,0,"Dear John, Dear Coltrane by Michael S. Harper",Michael S. Harper,"'Dear John, Dear Coltrane' by Michael S. Harpe...","a love supreme, a love supreme\na love supreme...",https://www.poetryfoundation.org/poems/42827/d...,"The poem explores themes of love, loss, pain, ...",A lone musician stands on a dimly lit stage un...,videos/0.mp4,"A person stands in a bustling marketplace, sur..."
1,1,Parrot by Stevie Smith,Stevie Smith,‘Parrot‘ depicts the declining health of a won...,The old sick green parrot\nHigh in a dingy cag...,https://revise.wales/pastPapers/A-level/Englis...,"This old parrot, sick and full of rage, longs ...",A weathered and aged parrot with vibrant yet m...,videos/1.mp4,"Scene 1: A sick green parrot with dull, dishev..."
2,2,Dust of Snow by Robert Frost,Robert Frost,"The simplicity, in the end, is the key element...",The way a crow\nShook down on me\nThe dust of ...,https://www.poetryfoundation.org/poems/44262/d...,The sight of a crow shaking snow from a tree t...,"A solitary poet, bundled in a woolen coat and ...",videos/2.mp4,"A person stands in a quiet, natural setting su..."
3,3,Suburban Sonnet by Gwen Harwood,Gwen Harwood,'Suburban Sonnet' by Gwen Harwood is a poem ab...,"She practises a fugue, though it can matter\nt...",https://genius.com/Gwen-harwood-suburban-sonne...,"A mother practices music, but her children int...",A mother sits at a grand piano in a cozy livin...,videos/3.mp4,Scene 1:\nA woman practices playing a keyboard...
4,4,Unending Love by Rabindranath Tagore,Rabindranath Tagore,'Unending Love' by Rabindranath Tagore is a he...,"I seem to have loved you in numberless forms, ...",https://allpoetry.com/Unending-Love,The speaker expresses their eternal love for s...,A person stands in a serene open field under a...,videos/4.mp4,A serene stream flows gently under a starry ni...


In [15]:
prompt = df.iloc[2]["prompt_by_ctext"]
print(prompt)
generate_video(prompt, "videos2/try.mp4", pipe=pipe,seconds=6,fps=6)

A person stands in a quiet, natural setting surrounded by tall hemlock trees, their head slightly tilted upward as they observe a crow perched on a branch. The crow moves suddenly, shaking a small amount of snow off the branch, and tiny snowflakes fall onto the person below. The person remains still, standing amidst the serene winter landscape with snow covering parts of the ground and tree branches. They appear to be wearing a simple, dark winter coat, and their breath is visible in the cold air. The scene captures the stillness of the moment as the snow gently falls around the person and the crow flies off into the distance.
Generating 32 frames


  0%|          | 0/50 [00:00<?, ?it/s]

Decoding...


'videos2/try.mp4'

In [43]:
id=4
print(df.iloc[id]["ctext"])
print("+"*10)
print(df.iloc[id]["prompt_by_ctext"])
print("+"*10)
Video("videos/"+str(id)+".mp4")

I seem to have loved you in numberless forms, numberless times…
In life after life, in age after age, forever.
My spellbound heart has made and remade the necklace of songs,
That you take as a gift, wear round your neck in your many forms,
In life after life, in age after age, forever.

Whenever I hear old chronicles of love, its age-old pain,
Its ancient tale of being apart or together.
As I stare on and on into the past, in the end you emerge,
Clad in the light of a pole-star piercing the darkness of time:
You become an image of what is remembered forever.

You and I have floated here on the stream that brings from the fount.
At the heart of time, love of one for another.
We have played alongside millions of lovers, shared in the same
Shy sweetness of meeting, the same distressful tears of farewell-
Old love but in shapes that renew and renew forever.

Today it is heaped at your feet, it has found its end in you
The love of all man’s days both past and forever:
Universal joy, univers

In [42]:
Video("videos2/"+str(id)+".mp4")

In [18]:
def process_dataset(
    df,
    pipe,
    video_dir: str = "videos2",
    input_col: str = "prompt_by_ctext",
    output_col: str = "video_by_poem",
    id_col: str = "id",
    csv_path: str = csv_path,
    batch_size: int = 5
):
    import os

    # Ensure output column exists
    if output_col not in df.columns:
        df[output_col] = ""

    # Select only rows that need processing
    pending = df[df[output_col].isna() | (df[output_col].astype(str).str.strip() == "")]
    
    if len(pending) == 0:
        print("All videos already generated.")
        return

    print(f"Pending videos: {len(pending)}")

    # Process in batches
    for start in range(0, len(pending), batch_size):
        end = start + batch_size
        batch = pending.iloc[start:end]

        print(f"\nProcessing batch {start} → {min(end, len(pending))}")

        for idx, row in batch.iterrows():
            poem_id = row[id_col]
            prompt = row[input_col]
            save_path = os.path.join(video_dir, f"{poem_id}.mp4")

            print(f"{idx} ----------------------------------------------------")

            try:
                generate_video(prompt, save_path, pipe)
                df.loc[df[id_col] == poem_id, output_col] = save_path
            except Exception as e:
                print(f"Error on {poem_id}: {e}")

        # Save progress after each batch
        df.to_csv(csv_path, index=False)
        print("Batch saved.")

    print("Completed all videos.")


In [20]:
process_dataset(
    df,
    pipe,
    video_dir="videos2",
    input_col="prompt_by_ctext",
    output_col="video_by_poem",
    batch_size=5
)

Pending videos: 150

Processing batch 0 → 5
0 ----------------------------------------------------
Generating 32 frames


  0%|          | 0/50 [00:00<?, ?it/s]

Decoding...
1 ----------------------------------------------------
Generating 32 frames


  0%|          | 0/50 [00:00<?, ?it/s]

Decoding...
2 ----------------------------------------------------
Generating 32 frames


...

Decoding...
Batch saved.

Processing batch 10 → 15
10 ----------------------------------------------------
Generating 32 frames


  0%|          | 0/50 [00:00<?, ?it/s]

Decoding...
11 ----------------------------------------------------
Generating 32 frames


  0%|          | 0/50 [00:00<?, ?it/s]

KeyboardInterrupt: 